# LLM-Based Request Understanding

This notebook demonstrates how a language model converts a natural-language weather question into structured request fields, including activity, location, date, time period, and optional elevation.

It also explores how conversation context can help resolve follow-up questions. The notebook uses an existing language model; it does not train a new model.


### Shared project OpenAI client
Run `00_Project_Setup.ipynb` once first. This notebook then uses the shared project configuration; no API key is entered here.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


### Project-level LLM client
This notebook uses the API configuration initialized in `00_Project_Setup.ipynb`. No API key is entered here.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.services.llm_service import get_client, MODEL_NAME

client = get_client()
print('Shared OpenAI client ready.')


Shared OpenAI client ready.


## 1. Install the OpenAI Python package

Run this cell once in Google Colab.

In [3]:
%pip -q install -U openai

DEPRECATION: Loading egg at /Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/jupyter-1.0.0-py3.11.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Configure the API key securely

For the POC, use a Colab Secret named:

`OPENAI_API_KEY`

Do **not** write your API key directly into the notebook.

## 3. Initialize the LLM client

The model name is kept in one variable so it can be changed without modifying the rest of the notebook.

## 4. Define the structured request format

The agent should return a predictable structure that the Weather Tool can use.

Example:

```python
{
    "activity": "cycling",
    "location": "Berlin",
    "date": "tomorrow",
    "time_period": "afternoon",
    "elevation_m": None
}
```

In [4]:
REQUEST_SCHEMA = {
    "type": "object",
    "properties": {
        "activity": {
            "type": "string",
            "description": "Outdoor activity requested by the user."
        },
        "location": {
            "type": "string",
            "description": "Place for which weather is requested."
        },
        "date": {
            "type": "string",
            "description": "Requested date expressed as the user stated it."
        },
        "time_period": {
            "type": "string",
            "description": (
                "Requested time such as morning, afternoon, evening, "
                "overnight, or a specific time. Use an empty string if absent."
            )
        },
        "elevation_m": {
            "type": ["number", "null"],
            "description": "Requested elevation in metres, if explicitly provided."
        }
    },
    "required": [
        "activity",
        "location",
        "date",
        "time_period",
        "elevation_m"
    ],
    "additionalProperties": False
}

## 5. Agent instructions

The system instructions define what the LLM is responsible for.

The agent should extract what the user actually says and should not invent missing information.

In [5]:
SYSTEM_PROMPT = """
You are the Weather Forecast Chatbot's request-understanding agent.

Your task is to extract structured information from a user's natural-language
outdoor weather request.

Extract:
- activity
- location
- date
- time period
- elevation in metres when explicitly provided

Supported activities include:
hiking, cycling, mountain biking, running, camping,
trail running, and outdoor tour.

Rules:
1. Extract information from the user's wording.
2. Do not invent a location, date, time, activity, or elevation.
3. Preserve relative date wording such as "tomorrow", "Saturday", or "this weekend".
4. If a time period is not given, return an empty string.
5. If elevation is not given, return null.
6. Return only the requested structured data.
"""

## 6. Create the extraction function

In [6]:
import json

def extract_weather_request(user_question):
    response = client.responses.create(
        model=MODEL_NAME,
        instructions=SYSTEM_PROMPT,
        input=user_question,
        text={
            "format": {
                "type": "json_schema",
                "name": "weather_request",
                "schema": REQUEST_SCHEMA,
                "strict": True
            }
        }
    )

    return json.loads(response.output_text)

## 7. Test the first example question

An example request is:

> “What’s the weather like for my hike in Munich tomorrow?”

We test whether the LLM can convert this into structured parameters.

In [7]:
question_1 = "What's the weather like for my hike in Munich tomorrow?"

request_1 = extract_weather_request(question_1)

request_1

{'activity': 'hiking',
 'location': 'Munich',
 'date': 'tomorrow',
 'time_period': '',
 'elevation_m': None}

## 8. Test the cycling scenario

In [8]:
question_2 = (
    "Will it rain during my cycling tour in Berlin tomorrow afternoon?"
)

request_2 = extract_weather_request(question_2)

request_2

{'activity': 'cycling',
 'location': 'Berlin',
 'date': 'tomorrow',
 'time_period': 'afternoon',
 'elevation_m': None}

## 9. Test the elevation scenario

An example request includes a hiking request near Innsbruck at approximately 2,500 metres.

In [9]:
question_3 = (
    "I'm hiking near Innsbruck at about 2,500 metres tomorrow. "
    "Will it be colder and windier?"
)

request_3 = extract_weather_request(question_3)

request_3

{'activity': 'hiking',
 'location': 'near Innsbruck',
 'date': 'tomorrow',
 'time_period': '',
 'elevation_m': 2500}

## 10. Batch-test the ten example questions

In [10]:
test_questions = [
    "What's the weather like for my hike in Munich tomorrow?",
    "Will it rain during my cycling tour in Berlin tomorrow afternoon?",
    "Is Saturday morning a good time for mountain biking in Garmisch-Partenkirchen?",
    "When would be the best time to go running in Cologne tomorrow?",
    "I'm camping near Lake Garda this weekend. What will the overnight weather be like?",
    "I'm hiking near Innsbruck at about 2,500 metres tomorrow. Will it be colder and windier?",
    "I want to cycle through the Black Forest tomorrow. Will strong wind or rain be a problem?",
    "I'm planning a trail run in the Dolomites on Sunday. Should I expect rain or snow?",
    "Will I have clear conditions for my hike near Salzburg on Saturday morning?",
    "I'm planning an outdoor tour in Hamburg this weekend. Which day and time look most suitable?"
]

extracted_requests = []

for i, question in enumerate(test_questions, start=1):
    result = extract_weather_request(question)

    extracted_requests.append({
        "Question": i,
        "User input": question,
        "Extracted request": result
    })

extracted_requests

[{'Question': 1,
  'User input': "What's the weather like for my hike in Munich tomorrow?",
  'Extracted request': {'activity': 'hiking',
   'location': 'Munich',
   'date': 'tomorrow',
   'time_period': '',
   'elevation_m': None}},
 {'Question': 2,
  'User input': 'Will it rain during my cycling tour in Berlin tomorrow afternoon?',
  'Extracted request': {'activity': 'cycling',
   'location': 'Berlin',
   'date': 'tomorrow',
   'time_period': 'afternoon',
   'elevation_m': None}},
 {'Question': 3,
  'User input': 'Is Saturday morning a good time for mountain biking in Garmisch-Partenkirchen?',
  'Extracted request': {'activity': 'mountain biking',
   'location': 'Garmisch-Partenkirchen',
   'date': 'Saturday',
   'time_period': 'morning',
   'elevation_m': None}},
 {'Question': 4,
  'User input': 'When would be the best time to go running in Cologne tomorrow?',
  'Extracted request': {'activity': 'running',
   'location': 'Cologne',
   'date': 'tomorrow',
   'time_period': '',
   'el

## 11. Convert extraction results to a table

In [11]:
import pandas as pd

evaluation_rows = []

for item in extracted_requests:
    request = item["Extracted request"]

    evaluation_rows.append({
        "Question": item["Question"],
        "Activity": request["activity"],
        "Location": request["location"],
        "Date": request["date"],
        "Time period": request["time_period"],
        "Elevation (m)": request["elevation_m"]
    })

evaluation_df = pd.DataFrame(evaluation_rows)

evaluation_df

,Question,Activity,Location,Date,Time period,Elevation (m)
0,1,hiking,Munich,tomorrow,,NaN
1,2,cycling,Berlin,tomorrow,afternoon,NaN
2,3,mountain biking,Garmisch-Partenkirchen,Saturday,morning,NaN
3,4,running,Cologne,tomorrow,,NaN
4,5,camping,near Lake Garda,this weekend,overnight,NaN
5,6,hiking,near Innsbruck,tomorrow,,2500.0
6,7,cycling,Black Forest,tomorrow,,NaN
7,8,trail running,the Dolomites,Sunday,,NaN
8,9,hiking,near Salzburg,Saturday,morning,NaN
9,10,outdoor tour,Hamburg,this weekend,,NaN


## 12. Architecture summary

The LLM component now performs the first part of the request-processing workflow:

```text
Natural-language question
          ↓
     LLM Agent
          ↓
Structured parameters
          ↓
activity / location / date /
time / elevation
```

### Next integration step

The next version will connect these extracted parameters to:

1. the location geocoder;
2. the Open-Meteo Weather Tool;
3. the Activity Weather Logic;
4. an LLM response-generation step.

That will produce the first complete end-to-end chatbot response.

### Security note

The API key should never be committed to GitHub, uploaded in the final project ZIP, or included directly in source code.